In [ ]:
%pip install anthropic

In [ ]:
import anthropic
from dotenv import load_dotenv
import os
import json
from datetime import datetime
load_dotenv()
client = anthropic.Anthropic()
TRANSACTIONS_FILE = "transactions.json"
messages=[]


tools=[{"name":"add_transaction","description":"Add a transaction to my budget losses such as buying a burger",        
        "input_schema":{
        "type": "object",
        "properties": {
            "amount": {"type": "number", "description": "Price of the good being purchased"},
            "category": {"type": "string", "description": "Category of good being purchased, current categories to use are [Item, Food and Service]"},
            "description": {"type": "string", "description": "Description of good being purchased"},
            "datetoday": {"type": "string", "description": "Date of purchase in YYYY-MM-DD format. Leave out to use today's date,  remember the year currently is 2026"}
        },
            "required":["amount","category","description"]
        }},
      {"name":"list_transactions","description":"Find a list of objects in a certain category",        
        "input_schema":{
        "type": "object",
        "properties": {
            "category": {"type": "string", "description": "List of transactions by category, Leave out category parameter if you want all transactions"},
        },
            "required":[]
        }},
      {"name":"delete_transaction","description":"Delete transaction of certain id",        
        "input_schema":{
        "type": "object",
        "properties": {
            "transaction_id": {"type": "number", "description": "Id of the transaction you want to delete"},
        },
            "required":["transaction_id"]
        }}
]

def load_transactions():
    if os.path.exists(TRANSACTIONS_FILE):
        with open(TRANSACTIONS_FILE, 'r') as f:
            transactions = json.load(f)
        return transactions
    else:
        return []

def save_transactions(transactions):
    with open(TRANSACTIONS_FILE, 'w') as f:
        json.dump(transactions,f,indent=2)

def add_transaction(amount, category, description, datetoday=None):
    if datetoday is None:
        datetoday = datetime.now().date().isoformat()
    data = load_transactions()
    highest = 0
    for i in data:
        if highest<i["id"]:
            highest=i["id"]
    new_id = highest + 1
    transaction = {
        "id":new_id,
        "amount":amount,
        "category":category,
        "description":description,
        "date":datetoday
    }
    data.append(transaction)
    save_transactions(data)
    return transaction

def list_transactions(category=None):
    data = load_transactions()
    if category is None:
        return data
    returntrans = []
    for i in data:
        if i["category"].lower()==category.lower():
            returntrans.append(i)
    return returntrans

def delete_transaction(transaction_id):
    data = load_transactions()
    deleted = None
    kept = []
    for i in data:
        if i["id"] == transaction_id:
            deleted = i
        else:
            kept.append(i)
    if deleted is None:
        return{"Error": "No object found on this id for deletion"}
    save_transactions(kept)
    return deleted

tool_functions = {"add_transaction":add_transaction,"list_transactions":list_transactions,"delete_transaction":delete_transaction}
while True:
    user_text = input()
    if user_text == "exit":
        break
    messages.append({"role":"user","content":user_text})
    message = client.messages.create(
        tools=tools,
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=messages
    )
    messages.append({"role":"assistant","content":message.content})
    while message.stop_reason == "tool_use":
        tool_results = []
        for i in message.content:
            if i.type == "tool_use":
                result = tool_functions[i.name](**i.input)
                tool_results.append({
                    "type":"tool_result",
                    "tool_use_id":i.id,
                    "content":json.dumps(result)
                })
        messages.append({"role":"user","content":tool_results})
        message = client.messages.create(
            tools=tools,
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            messages=messages
        )
        messages.append({"role": "assistant", "content": message.content})
        
    for block in message.content:
        if block.type == "text":
            print(block.text)


 I bought a Marvin Gaye Vinyl for 45 bucks


Got it! I've added your Marvin Gaye Vinyl purchase to your budget:
- **Amount:** $45.00
- **Category:** Item
- **Description:** Marvin Gaye Vinyl
- **Date:** September 28, 2026


 Remove it


Done! I've removed the Marvin Gaye Vinyl transaction from your budget.


 Add it again


Done! I've added the Marvin Gaye Vinyl back to your budget:
- **Amount:** $45.00
- **Category:** Item
- **Description:** Marvin Gaye Vinyl
- **Date:** September 28, 2026
